In [1]:
import torch
import numpy
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd
import nltk
# from nltk.tokenize import word_tokenize
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from torch.utils.data import DataLoader,TensorDataset
from torch.nn.utils.rnn import pad_sequence
import math
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
algebra_test=pd.read_parquet('hendrycks_math/algebra/test-00000-of-00001.parquet')
count_and_prob_test=pd.read_parquet('hendrycks_math/counting_and_probability/test-00000-of-00001.parquet')
geometry_test=pd.read_parquet('hendrycks_math/geometry/test-00000-of-00001.parquet')
intermediate_algebra_test=pd.read_parquet('hendrycks_math/intermediate_algebra/test-00000-of-00001.parquet')
number_theory_test=pd.read_parquet('hendrycks_math/number_theory/test-00000-of-00001.parquet')
prealgebra_test=pd.read_parquet('hendrycks_math/prealgebra/test-00000-of-00001.parquet')
precalculus_test=pd.read_parquet('hendrycks_math/precalculus/test-00000-of-00001.parquet')


In [3]:
algebra_train=pd.read_parquet('hendrycks_math/algebra/train-00000-of-00001.parquet')
count_and_prob_train=pd.read_parquet('hendrycks_math/counting_and_probability/train-00000-of-00001.parquet')
geometry_train=pd.read_parquet('hendrycks_math/geometry/train-00000-of-00001.parquet')
intermediate_algebra_train=pd.read_parquet('hendrycks_math/intermediate_algebra/train-00000-of-00001.parquet')
number_theory_train=pd.read_parquet('hendrycks_math/number_theory/train-00000-of-00001.parquet')
prealgebra_train=pd.read_parquet('hendrycks_math/prealgebra/train-00000-of-00001.parquet')
precalculus_train=pd.read_parquet('hendrycks_math/precalculus/train-00000-of-00001.parquet')


In [4]:
ds_test=[algebra_test,count_and_prob_test,geometry_test,intermediate_algebra_test,number_theory_test,prealgebra_test,precalculus_test]
ds_train=[algebra_train,count_and_prob_train,geometry_train,intermediate_algebra_train,number_theory_train,prealgebra_train,precalculus_train]

In [5]:
final_test=pd.concat(ds_test,ignore_index=True)
final_train=pd.concat(ds_train,ignore_index=True)


In [6]:
final_x_test=final_test.iloc[: ,0]
final_y_test=final_test.iloc[:, -1]
final_y_test
final_x_test

0       How many vertical asymptotes does the graph of...
1       What is the positive difference between $120\%...
2       Find $x$ such that $\lceil x \rceil + x = \dfr...
3                          Evaluate $i^5+i^{-25}+i^{45}$.
4                 If $2^8=4^x$, what is the value of $x$?
                              ...                        
4995    If $\sin x + \cos x = \frac{1}{5}$ and $0 < x ...
4996    The matrix for projecting onto a certain plane...
4997    Let $\mathbf{a},$ $\mathbf{b},$ and $\mathbf{c...
4998    Find the smallest positive integer solution to...
4999    The matrix for reflecting over a certain line ...
Name: problem, Length: 5000, dtype: str

In [7]:
final_x_train=final_train.iloc[:,0]
final_y_train=final_train.iloc[:, -1]
final_y_train
final_x_train

0       Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...
1       A rectangular band formation is a formation wi...
2       What is the degree of the polynomial $(4 +5x^3...
3       Evaluate $\left\lceil3\left(6-\frac12\right)\r...
4       Sam is hired for a 20-day period. On days that...
                              ...                        
7495    Find the number of real solutions of the equat...
7496    Let $A,$ $B,$ $C$ be the angles of a triangle....
7497    Let $G$ be the centroid of triangle $ABC,$ and...
7498    If angle $A$ lies in the second quadrant and $...
7499    The real numbers $a$ and $b$ satisfy\n\[\begin...
Name: problem, Length: 7500, dtype: str

In [8]:
tokenizer=Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer=Whitespace()
trainer=BpeTrainer(special_tokens=["[UNK]","[CLS]","[SEP]","[PAD]","[MASK]"],vocab_size=1000)
tokenizer.train_from_iterator(final_x_train,trainer)


x_train_tokenized = [tokenizer.encode(text) for text in final_x_train]
y_train_tokenized = [tokenizer.encode(text) for text in final_y_train]

x_test_tokenized  = [tokenizer.encode(text) for text in final_x_test]
y_test_tokenized  = [tokenizer.encode(text) for text in final_y_test]





In [9]:
y_train_tokenized[1].ids

[267,
 8,
 92,
 8,
 115,
 101,
 175,
 107,
 70,
 114,
 158,
 996,
 87,
 100,
 329,
 326,
 183,
 101,
 791,
 112,
 377,
 300,
 16,
 475,
 274,
 126,
 365,
 286,
 18,
 534,
 113,
 577,
 339,
 706,
 123,
 274,
 428,
 87,
 361,
 101,
 972,
 100,
 377,
 300,
 30,
 639,
 86,
 92,
 15,
 22,
 33,
 81,
 639,
 8,
 254,
 86,
 17,
 22,
 525,
 92,
 15,
 21,
 627,
 81,
 639,
 55,
 208,
 983,
 745,
 491,
 16,
 577,
 470,
 30,
 639,
 86,
 92,
 15,
 22,
 343,
 86,
 17,
 22,
 525,
 92,
 15,
 21,
 627,
 86,
 92,
 17,
 420,
 15,
 86,
 17,
 22,
 639,
 639,
 22,
 930,
 420,
 15,
 86,
 17,
 22,
 639,
 639,
 24,
 33,
 86,
 17,
 420,
 639,
 59,
 73,
 79,
 82,
 167,
 159,
 101,
 70,
 114,
 259,
 679,
 416,
 471,
 158,
 996,
 87,
 18,
 38,
 117,
 220,
 111,
 101,
 541,
 428,
 16,
 577,
 881,
 372,
 8,
 86,
 92,
 8,
 679,
 416,
 29,
 28,
 18,
 59,
 73,
 339,
 75,
 89,
 120,
 87,
 114,
 134,
 73,
 205,
 834,
 451,
 107,
 8,
 86,
 8,
 114,
 8,
 92,
 8,
 100,
 101,
 80,
 461,
 428,
 18,
 246,
 8,
 86,
 33,
 425,
 15

In [10]:
y_train_ids=[torch.tensor(item.ids) for item in y_train_tokenized]
x_train_ids=[torch.tensor(item.ids) for item in x_train_tokenized]
x_test_ids=[torch.tensor(item.ids) for item in x_test_tokenized]
y_test_ids=[torch.tensor(item.ids) for item in x_test_tokenized]



In [11]:
pad_id = tokenizer.token_to_id("[PAD]")

In [12]:
padded_x_train=pad_sequence(x_train_ids,batch_first=True,padding_value=pad_id)
padded_y_train=pad_sequence(y_train_ids,batch_first=True,padding_value=pad_id)
padded_x_test=pad_sequence(x_test_ids,batch_first=True,padding_value=pad_id)
padded_y_test=pad_sequence(y_test_ids,batch_first=True,padding_value=pad_id)



In [13]:
padded_x_train

tensor([[267, 232,  74,  ...,   3,   3,   3],
        [ 37, 963,  70,  ...,   3,   3,   3],
        [168, 108, 101,  ...,   3,   3,   3],
        ...,
        [267,   8,  43,  ...,   3,   3,   3],
        [246, 179,   8,  ...,   3,   3,   3],
        [195, 240, 288,  ...,   3,   3,   3]])

In [34]:
train_batch_ids=TensorDataset(padded_x_train,padded_y_train)
train_dataloader=DataLoader(dataset=train_batch_ids,batch_size=4,shuffle=True,num_workers=4)
train_dataloader
test_batch_ids=TensorDataset(padded_x_test,padded_y_test)
test_dataloader=DataLoader(dataset=test_batch_ids,batch_size=4,shuffle=True,num_workers=4)



In [35]:
class transformerembeddinfs(nn.Module):
    def __init__(self,vocab_size,d_model,max_len):
        super().__init__()
        self.d_model=d_model
        self.token_embedding=nn.Embedding(vocab_size,d_model)

        pe=torch.zeros(max_len,d_model)
        position=torch.arange(0,max_len,dtype=torch.float).unsqueeze(1)
        div_term=torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:,0::2]=torch.sin(position*div_term)
        pe[:,1::2]=torch.cos(position*div_term)
        pe=pe.unsqueeze(0)
        self.register_buffer("pe",pe)
    def forward(self,x):
        seq_len=x.size(1)
        tok_emb = self.token_embedding(x) * math.sqrt(self.d_model)
        pos_emb = self.pe[:, :seq_len, :]
        return tok_emb + pos_emb

attention=(b,seq,d_model)
b=no of batches
l=no of tokens per layer
d_model=no of features per model
num_head no of attention heads
d_k= features per head 
d_k=d_model/num_heads

In [36]:
class self_attention(nn.Module):
    def __init__(self,d_model,num_heads):
        super().__init__()
        assert d_model%num_heads==0, "no of heads must be divisible"
        self.d_model=d_model
        self.num_heads=num_heads
        self.d_k=d_model//num_heads
        self.w_q=nn.Linear(d_model,d_model,bias=False)
        self.w_k=nn.Linear(d_model,d_model,bias=False)
        self.w_v=nn.Linear(d_model,d_model,bias=False)
        self.w_o = nn.Linear(d_model, d_model)
    def scaled_prod(self, Q, K, V, mask=None):
        attention_scores = torch.matmul(
        Q, K.transpose(-2, -1)
        ) / math.sqrt(self.d_k)

        if mask is not None:
            attention_scores = attention_scores.masked_fill(
            mask == 0, -1e9
        )

        probabilities = torch.softmax(attention_scores, dim=-1)

        final_output = torch.matmul(probabilities, V)

        return final_output
    def split_heads(self,x):        
        batch,l,d_model=x.size()
        return x.view(batch,l,self.num_heads,self.d_k).transpose(1,2)
    def concat_heads(self,x):
        batch_size,_,seq_length,d_k=x.size()
        return x.transpose(1,2).contiguous().view(batch_size,seq_length,self.d_model)
    def forward(self,Q,K,V,mask=None):
        Q=self.split_heads(self.w_q(Q))
        K=self.split_heads(self.w_k(K))
        V=self.split_heads(self.w_v(V))
        attn_output=self.scaled_prod(Q,K,V,mask)
        output=self.w_o(self.concat_heads(attn_output))
        return output
        
            


In [37]:
class positionwisefeedfroward(nn.Module):
    def __init__(self,d_model,d_ff):
        super().__init__()
        self.fc1=nn.Linear(d_model,d_ff)
        self.fc2=nn.Linear(d_ff,d_model)
        self.relu=nn.ReLU()
    def forward(self,x):
        return(self.fc2(self.relu(self.fc1(x))))
    

In [38]:
class EncoderLayer(nn.Module):
    def __init__(self,d_model,num_heads,d_ff,dropout):
        super().__init__()
        self.self_attn=self_attention(d_model,num_heads)
        self.feedforward=positionwisefeedfroward(d_model,d_ff)
        self.layernorm1=nn.LayerNorm(d_model)
        self.layernorm2=nn.LayerNorm(d_model)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x,mask):
        attn_output=self.self_attn(x,x,x,mask)
        x=self.layernorm1(x+self.dropout(attn_output))
        ff_output=self.feedforward(x)
        x=self.layernorm2(x+self.dropout(ff_output))
        return x

In [45]:
class decoderlayer(nn.Module):
    def __init__(self,d_model,num_heads,d_ff,dropout):
        super().__init__()
        self.self_attn=self_attention(d_model,num_heads)
        self.cross_attn=self_attention(d_model,num_heads)
        self.feedforward=positionwisefeedfroward(d_model,d_ff)
        self.layernorm1=nn.LayerNorm(d_model)
        self.layernorm2=nn.LayerNorm(d_model)
        self.layernorm3=nn.LayerNorm(d_model)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x,enc_output,src_mask,tgt_mask):
        attn_output=self.self_attn(x,x,x,tgt_mask)
        x=self.layernorm1(x+self.dropout(attn_output))
        attn_output=self.cross_attn(x,enc_output,enc_output,src_mask)
        x=self.layernorm2(x+self.dropout(attn_output))
        ff_output=self.feedforward(x)
        x=self.layernorm3(x+self.dropout(ff_output))
        return x

In [46]:
class transformer(nn.Module):
    def __init__(self, src_vocab_size,tgt_vocab_size,d_model,num_heads,num_layers,d_ff,max_seq_length,dropout):
        super().__init__()
        self.encoder_embedding=transformerembeddinfs(src_vocab_size,d_model,max_seq_length)
        self.pad_id=pad_id
        self.decoder_embedding=transformerembeddinfs(src_vocab_size,d_model,max_seq_length)
        self.encode_layers=nn.ModuleList([EncoderLayer(d_model,num_heads,d_ff,dropout)for _ in range(num_layers)])
        self.decode_layers=nn.ModuleList([decoderlayer(d_model,num_heads,d_ff,dropout)for _ in range(num_layers)])
        self.fc=nn.Linear(d_model,tgt_vocab_size)
        self.dropout=nn.Dropout()
    def generate_mask(self, src, tgt):
        src_mask = (src != self.pad_id).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != self.pad_id).unsqueeze(1).unsqueeze(2)

        seq_length = tgt.size(1)

        nopeak_mask = (
        1 - torch.triu(
            torch.ones(
                1, seq_length, seq_length,
                device=tgt.device
            ),
            diagonal=1
        )
        ).bool()

        tgt_mask = tgt_mask & nopeak_mask

        return src_mask, tgt_mask
    def forward(self,src,tgt):
        src_mask,tgt_mask=self.generate_mask(src,tgt)
        src_embedded = self.dropout(self.encoder_embedding(src))
        tgt_embedded = self.dropout(self.decoder_embedding(tgt))

        enc_output = src_embedded
        for enc_layer in self.encode_layers:
            enc_output = enc_layer(enc_output, src_mask)

        dec_output = tgt_embedded
        for dec_layer in self.decode_layers:
            dec_output = dec_layer(dec_output, enc_output, src_mask, tgt_mask)

        output = self.fc(dec_output)
        return output


In [48]:
from tqdm.auto import tqdm

In [49]:
# Vocabulary sizes
src_vocab_size = tokenizer.get_vocab_size()
tgt_vocab_size = tokenizer.get_vocab_size()

# Transformer hyperparameters
d_model = 128
num_heads = 4
num_layers = 2
d_ff = 512
max_seq_length = max(
    padded_x_train.size(1),
    padded_y_train.size(1)
)
dropout = 0.1

# Create the model
model = transformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    num_layers=num_layers,
    d_ff=d_ff,
    max_seq_length=max_seq_length,
    dropout=dropout
)
model=model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001,
    betas=(0.9, 0.98),
    eps=1e-9
)


num_epochs = 10
total_samples = len(train_dataloader.dataset)
samples_processed = 0

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    samples_processed_epoch = 0

    progress_bar = tqdm(
        train_dataloader,
        desc=f"Epoch {epoch + 1}/{num_epochs}",
        unit="batch"
    )

    for src_data, tgt_data in progress_bar:
        src_data = src_data.to(device)
        tgt_data = tgt_data.to(device)

        optimizer.zero_grad()

        output = model(src_data, tgt_data[:, :-1])
        targets = tgt_data[:, 1:]

        loss = criterion(
            output.reshape(-1, tgt_vocab_size),
            targets.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        batch_size = src_data.size(0)
        samples_processed += batch_size
        samples_processed_epoch += batch_size
        total_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            samples=f"{samples_processed_epoch}/{total_samples}"
        )

    avg_loss = total_loss / len(train_dataloader)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Average Loss: {avg_loss:.4f} | "
        f"Samples processed this epoch: "
        f"{samples_processed_epoch}/{total_samples} | "
        f"Cumulative samples processed: {samples_processed}"
    )


Epoch 1/10: 100%|██████████| 1875/1875 [26:56<00:00,  1.16batch/s, loss=4.6198, samples=7500/7500]


Epoch 1/10 | Average Loss: 4.9523 | Samples processed this epoch: 7500/7500 | Cumulative samples processed: 7500


Epoch 2/10: 100%|██████████| 1875/1875 [25:22<00:00,  1.23batch/s, loss=3.3220, samples=7500/7500]


Epoch 2/10 | Average Loss: 4.1532 | Samples processed this epoch: 7500/7500 | Cumulative samples processed: 15000


Epoch 3/10: 100%|██████████| 1875/1875 [25:20<00:00,  1.23batch/s, loss=3.4178, samples=7500/7500]


Epoch 3/10 | Average Loss: 3.8682 | Samples processed this epoch: 7500/7500 | Cumulative samples processed: 22500


Epoch 4/10: 100%|██████████| 1875/1875 [41:54<00:00,  1.34s/batch, loss=3.8092, samples=7500/7500]


Epoch 4/10 | Average Loss: 3.7120 | Samples processed this epoch: 7500/7500 | Cumulative samples processed: 30000


Epoch 5/10:  39%|███▉      | 733/1875 [18:01<28:05,  1.48s/batch, loss=3.3160, samples=2932/7500]


KeyboardInterrupt: 

In [51]:
torch.save({
    "epoch": 4,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": 3.8092,
}, "transformer_epoch4_checkpoint.pth")

In [52]:
import os
print(os.path.abspath("transformer_epoch4_checkpoint.pth"))
print(os.path.getsize("transformer_epoch4_checkpoint.pth") / (1024**2), "MB")

c:\Users\HP\.vscode\python\ml\llms\adaptive_depth_slm_for maths\transformer_epoch4_checkpoint.pth
17.66431713104248 MB
